# 🤖 Phase 2: GenAI Frameworks & Building with LangChain + Groq
### Topics: Prompt Templates | Output Parsers | Chaining | Blog Builder Project

---
> **Note for students**: Run each cell in order from top to bottom. Never skip cells!

## ⬇️ Step 0: Install Libraries
We need the following packages:
- `langchain` — the main AI orchestration framework
- `langchain-groq` — Groq-specific integration for LangChain
- `python-dotenv` — loads secrets from a `.env` file

In [1]:
# Run this cell only once to install all required libraries
!pip install langchain langchain-groq python-dotenv

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip


## 🔐 Step 1: Setup API Key with .env

**⚠️ IMPORTANT**: Never put API keys directly in your code! Use a `.env` file.

**What is a `.env` file?**
A `.env` file is a plain text file that stores sensitive configuration values (like passwords and API keys) outside of your code. We use `python-dotenv` to load them into our program.

**How to set it up:**
1. Create a file called `.env` in the same directory as this notebook.
2. Add this line to it: `GROQ_API_KEY=your_actual_api_key_here`
3. Get a FREE API key from https://console.groq.com

**✅ How the code reads it:**

In [2]:
import os
from dotenv import load_dotenv

# This reads the .env file and loads GROQ_API_KEY into the system environment
load_dotenv()

# Verify the key is loaded (it will print 'True' if successful, 'False' if not found)
api_key = os.getenv("GROQ_API_KEY")
print("API Key Loaded:", api_key is not None)
print("First 5 chars:", api_key[:5] if api_key else "NOT FOUND")

API Key Loaded: False
First 5 chars: NOT FOUND


if ur using the google colab


In [3]:
import os
os.environ["GROQ_API_KEY"] = "your_api_key_here"

---
# 🧠 PART 1: Your First Groq Chatbot

**What is Groq?**
Groq is a company that makes AI inference very **fast** using specialized hardware chips. They give free access to open-source models like:
- `llama3-8b-8192` — Small, fast Meta Llama 3 model
- `llama3-70b-8192` — Bigger, smarter Meta Llama 3 model
- `mixtral-8x7b-32768` — Very powerful Mixtral model with 32k context window

**How does ChatGroq work internally?**
When you call `llm.invoke(message)`, it:
1. Takes your text message
2. Sends it via an HTTP request to Groq's fast inference server
3. The server runs the LLM and streams back the result
4. LangChain packages the response as an `AIMessage` object
5. You call `.content` to get the actual text string

In [ ]:
from langchain_groq import ChatGroq

# Initialize the LLM
llm = ChatGroq(
    model="llama3-8b-8192",  # Fast, free, and capable model
    temperature=0.7          # 0 = boring/predictable, 1 = creative/wild
)

print("LLM Ready:", llm.model_name)

In [ ]:
# SIMPLE CHATBOT - Single message
response = llm.invoke("What is LangChain? Answer in 2 sentences.")

# The response is an AIMessage object
print("Type of response:", type(response))
print("\nAnswer:")
print(response.content)

In [ ]:
# SYSTEM + USER MESSAGE CHATBOT
# In chat models, we can give a 'system' message to set the role/persona!
from langchain.schema import HumanMessage, SystemMessage

messages = [
    SystemMessage(content="You are a funny joke-telling robot. Always end responses with a robot sound like 'BZZZT!'"),
    HumanMessage(content="Tell me a joke about Python programming.")
]

response = llm.invoke(messages)
print(response.content)

In [ ]:
# INTERACTIVE MULTI-TURN CHATBOT (Conversation with memory)
# The key trick: we store all messages and send the full history each time!

from langchain.schema import HumanMessage, SystemMessage, AIMessage

chat_history = [
    SystemMessage(content="You are a helpful AI tutor teaching a student about Python programming.")
]

def chat(user_input):
    """Send a message, get a reply, and remember the conversation."""
    # Add the user's message to history
    chat_history.append(HumanMessage(content=user_input))
    
    # Send the ENTIRE history to the LLM (this is how memory works in LLMs!)
    response = llm.invoke(chat_history)
    
    # Add the AI's reply to history so future calls remember it
    chat_history.append(AIMessage(content=response.content))
    
    return response.content

# Test it!
print("Turn 1:")
print(chat("What is a Python list?"))
print("\nTurn 2 (AI remembers context!):")
print(chat("Can you show me how to sort it?"))

### 👨‍💻 Practice 1: Change the Persona
**Task**: Modify the System Message in the chatbot above to give it a new persona: 
- Make it talk like a **pirate** who explains coding concepts.
- Ask it: "What is a for loop in Python?"

**Hint**: Only change the `SystemMessage(content=...)` line.

In [ ]:
# Write your pirate chatbot here!


'''
# Solution:
pirate_history = [
    SystemMessage(content="You are a pirate who explains Python coding. Use pirate language: Arr, Ahoy, matey etc.")
]
pirate_history.append(HumanMessage(content="What is a for loop in Python?"))
response = llm.invoke(pirate_history)
print(response.content)
'''

---
# 🗒️ PART 2: Prompt Templates

**The Problem**: Imagine you want to ask the LLM about 100 different topics. Writing 100 different prompts by hand is silly. **Prompt Templates solve this!**

**How it works internally:**
- A template is a string with `{variable}` placeholders 
- `PromptTemplate.from_template(text)` detects the `{}` and registers them as `input_variables`
- `.format(variable=value)` does a standard Python string substitution
- `.invoke(dict)` automatically formats and returns a prompt string ready for the LLM

In [ ]:
from langchain.prompts import PromptTemplate

# ---- BASIC PROMPT TEMPLATE ----
template = PromptTemplate.from_template(
    "Explain {concept} to a {audience} in simple terms. Keep it under 3 sentences."
)

# See the internal variables it detected!
print("Input variables:", template.input_variables)

# Format for 3 different audiences (notice same template, different values!)
audiences = ["10-year-old", "software engineer", "grandparent"]
for audience in audiences:
    formatted = template.format(concept="Machine Learning", audience=audience)
    print(f"\n--- Prompt for: {audience} ---")
    print(formatted)

In [ ]:
# ---- TEMPLATE + LLM CALL ----
# Now let's actually send the formatted prompt to the LLM!
formatted_prompt = template.format(concept="Neural Networks", audience="high school student")
response = llm.invoke(formatted_prompt)
print(response.content)

In [ ]:
# ---- CHAT PROMPT TEMPLATE (with System message) ----
from langchain.prompts import ChatPromptTemplate

# This handles both system and human messages in one template!
chat_template = ChatPromptTemplate.from_messages([
    ("system", "You are an expert {domain} teacher. Always give examples."),
    ("human", "Teach me about: {topic}")
])

formatted_messages = chat_template.format_messages(
    domain="Data Science",
    topic="Pandas DataFrames"
)

print("Messages sent:", [m.type for m in formatted_messages])

response = llm.invoke(formatted_messages)
print("\nResponse:")
print(response.content)

### 👨‍💻 Practice 2: Quiz Generator Template
**Task**: Create a `ChatPromptTemplate` that takes `{subject}` and `{difficulty}` as variables.
- The system message should say: "You are a strict exam paper setter."
- The human message should ask: "Create 3 {difficulty} level multiple choice questions about {subject}."
- Then call the LLM with `subject="Python"` and `difficulty="medium"` and print the result.

In [ ]:
# Write your quiz generator template here


'''
# Solution:
quiz_template = ChatPromptTemplate.from_messages([
    ("system", "You are a strict exam paper setter."),
    ("human", "Create 3 {difficulty} level multiple choice questions about {subject}.")
])

messages = quiz_template.format_messages(subject="Python", difficulty="medium")
response = llm.invoke(messages)
print(response.content)
'''

---
# 📤 PART 3: Output Parsers

**The Problem**: LLMs return plain text. But real applications need **structured data** (Python lists, JSON dicts, etc.).

**How it works internally:**
1. The parser's `.get_format_instructions()` method gives you a text snippet that tells the LLM HOW to format its answer.
2. You inject those instructions into your prompt.
3. After the LLM responds, `.parse()` splits/parses the text and returns a proper Python object.

In [ ]:
# ---- CUSTOM PARSER: Comma Separated List ----
from langchain.schema import BaseOutputParser

class ListParser(BaseOutputParser):
    """Converts a comma-separated string into a Python list."""
    def parse(self, text: str):
        # Strip whitespace, then split on commas
        items = [item.strip() for item in text.strip().split(",")]
        return items

parser = ListParser()

# Test it manually first
raw = "Python, JavaScript, Java, Go, Rust"
result = parser.parse(raw)
print("Type:", type(result))
print("List:", result)
print("First item:", result[0])

In [ ]:
# ---- USE THE PARSER WITH THE LLM ----
# Ask the LLM for a comma-separated list, then parse it!
prompt = PromptTemplate.from_template(
    "List exactly 5 famous {category}. Respond ONLY as a comma-separated list. Nothing else."
)

formatted = prompt.format(category="AI companies")
raw_response = llm.invoke(formatted)
print("Raw response text:")
print(raw_response.content)

print("\nParsed as Python list:")
parsed_list = parser.parse(raw_response.content)
print(parsed_list)
print("Number of items:", len(parsed_list))

In [ ]:
# ---- BUILT-IN JSON PARSER ----
from langchain.output_parsers import StructuredOutputParser, ResponseSchema

# Define the schema (the structure we want the AI to return)
schemas = [
    ResponseSchema(name="title",   description="A catchy blog post title"),
    ResponseSchema(name="summary", description="A one-sentence blog post summary"),
    ResponseSchema(name="tags",    description="3 relevant comma-separated tags")
]

structured_parser = StructuredOutputParser.from_response_schemas(schemas)

# See the format instructions the parser generates!
format_instructions = structured_parser.get_format_instructions()
print("Format instructions given to LLM:")
print(format_instructions[:400], "...")

In [ ]:
# Now use those instructions in our prompt so the LLM formats correctly!
blog_idea_prompt = PromptTemplate.from_template(
    "Generate a blog post idea about {topic}.\n{format_instructions}"
)

formatted = blog_idea_prompt.format(
    topic="Artificial Intelligence in Education",
    format_instructions=format_instructions
)

raw_response = llm.invoke(formatted)
print("Raw LLM output:")
print(raw_response.content)

print("\nParsed as Python dict:")
parsed_dict = structured_parser.parse(raw_response.content)
print(parsed_dict)
print("\nTitle only:", parsed_dict["title"])

### 👨‍💻 Practice 3: Product Info Parser
**Task**: Use `StructuredOutputParser` to extract the following info about a product:
- `name` — the product name
- `price` — a realistic price in INR
- `pros` — two advantages, comma-separated
- `cons` — one disadvantage

Ask the LLM to generate product info for: `"iPhone 16 Pro"`

In [ ]:
# Write your product info parser here


'''
# Solution:
product_schemas = [
    ResponseSchema(name="name",  description="The product name"),
    ResponseSchema(name="price", description="A realistic price in INR"),
    ResponseSchema(name="pros",  description="Two advantages, comma-separated"),
    ResponseSchema(name="cons",  description="One main disadvantage")
]
product_parser = StructuredOutputParser.from_response_schemas(product_schemas)
fi = product_parser.get_format_instructions()

product_prompt = PromptTemplate.from_template(
    "Give info for this product: {product}\n{format_instructions}"
)
formatted = product_prompt.format(product="iPhone 16 Pro", format_instructions=fi)
raw = llm.invoke(formatted)
print(product_parser.parse(raw.content))
'''

---
# 🔗 PART 4: Chaining with LCEL (LangChain Expression Language)

**The Problem**: Every time we want to run Prompt → LLM → Parser, we're writing 3 separate lines. **Chains combine them into one object!**

**How LCEL works internally:**
The `|` pipe operator wraps each component in a `Runnable`. When you call `.invoke()` on the chain:
- The output of each step is automatically passed as input to the next step
- `PromptTemplate.invoke()` returns a `PromptValue`
- `ChatGroq.invoke(PromptValue)` returns an `AIMessage`
- `Parser.invoke(AIMessage)` extracts `.content` and returns parsed output

```
Input Dict → [PromptTemplate] → Prompt String → [ChatGroq] → AIMessage → [Parser] → Python Object
```

In [ ]:
# ---- BASIC CHAIN: Prompt | LLM ----
fact_prompt = PromptTemplate.from_template(
    "Tell me one interesting fact about {country}. Keep it to 1 sentence."
)

# Build the chain using the pipe operator |
fact_chain = fact_prompt | llm

print("Chain type:", type(fact_chain))

# invoke() runs ALL steps in one call!
result = fact_chain.invoke({"country": "India"})
print("\nResponse type:", type(result))
print("Fact:", result.content)

In [ ]:
# ---- FULL CHAIN: Prompt | LLM | Parser ----
from langchain_core.output_parsers import StrOutputParser

# StrOutputParser simply extracts .content from the AIMessage
str_parser = StrOutputParser()

joke_prompt = PromptTemplate.from_template(
    "Write ONE funny pun about {topic}. Just the pun, nothing else."
)

joke_chain = joke_prompt | llm | str_parser

# Now .invoke() returns a clean string instead of AIMessage!
joke = joke_chain.invoke({"topic": "programming bugs"})
print("Type:", type(joke))  # str, not AIMessage
print("Joke:", joke)

In [ ]:
# ---- BATCH PROCESSING WITH CHAINS ----
# One of the most powerful features: run the chain on MULTIPLE inputs at once!

countries = [
    {"country": "Japan"},
    {"country": "Brazil"},
    {"country": "Egypt"}
]

# .batch() sends all 3 requests efficiently
results = fact_chain.batch(countries)

for country, result in zip(countries, results):
    print(f"🏛️ {country['country']}: {result.content[:100]}...")

### 👨‍💻 Practice 4: Multi-Language Translator Chain
**Task**: Build a chain that takes `{text}` and `{language}` and translates the text into that language.
Use `StrOutputParser` to get a clean string output.
Test it on 3 different languages using `.batch()`.

In [ ]:
# Build your translator chain here


'''
# Solution:
from langchain_core.output_parsers import StrOutputParser

translate_prompt = PromptTemplate.from_template(
    "Translate this text to {language}: {text}. Reply with ONLY the translation."
)
translator = translate_prompt | llm | StrOutputParser()

inputs = [
    {"text": "Hello, how are you?", "language": "Tamil"},
    {"text": "Hello, how are you?", "language": "French"},
    {"text": "Hello, how are you?", "language": "Japanese"}
]

results = translator.batch(inputs)
for inp, res in zip(inputs, results):
    print(f"{inp['language']}: {res}")
'''

---
# 🚀 PART 5: FINAL PROJECT — Automated Blog Builder

**Architecture:**
```
[User Topic]
     ↓
[Chain 1: outline_chain]   → Generates 3-point outline
     ↓
[Chain 2: article_chain]   → Expands outline into 200-word article
     ↓
[Chain 3: hashtag_chain]   → Generates 5 social media hashtags
     ↓
   [Output]
```

We will build this step-by-step!

In [ ]:
from langchain_core.output_parsers import StrOutputParser

# ============================================
# CHAIN 1: Topic -> Blog Outline
# ============================================
outline_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a professional blog content strategist."),
    ("human", "Create a concise 3-point outline for a blog post on: {topic}. Number the points 1, 2, 3.")
])
outline_chain = outline_prompt | llm | StrOutputParser()

# ============================================
# CHAIN 2: Outline -> Full Article
# ============================================
article_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert blog writer. Write engaging, informative content."),
    ("human", "Expand this outline into a complete 200-word blog article:\n\n{outline}")
])
article_chain = article_prompt | llm | StrOutputParser()

# ============================================
# CHAIN 3: Article -> Hashtags
# ============================================
hashtag_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a social media marketing expert."),
    ("human", "Generate exactly 5 trending hashtags for this article:\n\n{article}\n\nOutput ONLY the hashtags, space-separated.")
])
hashtag_chain = hashtag_prompt | llm | StrOutputParser()

print("✅ All 3 chains are built!")

In [ ]:
# ============================================
# CONNECT ALL CHAINS INTO ONE PIPELINE
# ============================================

def generate_full_blog(topic):
    """Auto-generates a complete blog post with hashtags from just a topic."""
    
    print(f"📝 TOPIC: {topic}")
    print("=" * 60)
    
    # STEP 1: Generate outline
    print("\n📌 GENERATING OUTLINE...")
    outline = outline_chain.invoke({"topic": topic})
    print(outline)
    
    # STEP 2: Generate article from outline
    print("\n✍️ GENERATING ARTICLE...")
    article = article_chain.invoke({"outline": outline})
    print(article)
    
    # STEP 3: Generate hashtags from article
    print("\n🏷️ GENERATING HASHTAGS...")
    hashtags = hashtag_chain.invoke({"article": article})
    print(hashtags)
    
    print("\n" + "=" * 60)
    print("✅ BLOG POST COMPLETE!")
    
    return {"outline": outline, "article": article, "hashtags": hashtags}


# RUN IT!
blog = generate_full_blog("The Future of AI in Education")

In [ ]:
# ---- BONUS: Make it interactive! ----
# Uncomment to type your own topic

# user_topic = input("💬 Enter a blog topic: ")
# generate_full_blog(user_topic)

### 🎓 Final Homework Challenge
**Extend the Blog Builder** with a 4th chain that:
- Takes the `{article}` as input
- Generates a **Twitter/X thread** version of the article (5 tweets, numbered 1/5, 2/5, etc.)
- Each tweet must be under 280 characters

Then integrate it into `generate_full_blog()` as `# STEP 4`.


In [ ]:
# Build your Twitter thread chain here!


'''
# Solution:
twitter_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a viral Twitter/X content creator."),
    ("human", "Convert this article into a 5-tweet thread. Number each tweet (1/5, 2/5 etc). Keep each tweet under 280 chars.\n\n{article}")
])
twitter_chain = twitter_prompt | llm | StrOutputParser()

# Test it:
# thread = twitter_chain.invoke({"article": blog["article"]})
# print(thread)
'''